In [ ]:
import matplotlib.pyplot as plt

# Ensure plots render inline inside Jupyter
%matplotlib inline 

import json 

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

retirement_age = 42
input_file = "test_owning_50k.json"
output_file = "owning_dist_50k.json"

In [ ]:
mc_results = None
with open(input_file, encoding="utf-8") as f:
    mc_results = json.load(f)
perf_data = mc_results["perf_data"]
sim_data = mc_results["simulations"]
models = [m for m in sim_data.keys()]

initial_nav = mc_results["initial_nav"]
years = mc_results["years"]
paths = mc_results["total_paths"]

run_stats = {}
run_stats["initial_nav"] = initial_nav
run_stats["years_to_simulate"] = years
run_stats["total_paths"] = paths
run_stats["retirement_age"] = retirement_age
run_stats["results"] = {}

def annualize(ret, years):
    neg = np.sign(ret)
    ret = abs(ret)
    a = np.pow(ret + 1, 1 / years) - 1
    return float(neg * a)

for m in models:
    r = sim_data[m]
    run_stats["results"][m] = []
    for sim in r:
        yearly_spending = sim["spending"]
        allocation_str = f"{sim["equity"]*100.0:02.0f}-{sim["ladder"]*100.0:02.0f}"
        df_data = pd.DataFrame(sim["results"])
        df_data["Returns"] = (df_data["Terminal NAV"] - initial_nav) / initial_nav
        ret_mean = df_data["Returns"].mean()
        # Compute the Expected Shortfall 5 and 10 for ages ending in ruin.
        ruin_ages = {i: int(v) for (i, v) in enumerate(sim["ruin_histogram"]) if v > 0}
        v = list(ruin_ages.keys())
        f = list(ruin_ages.values())
        ruin_flat_data = np.repeat(v, f)
        p5, p10, p50 = np.quantile(ruin_flat_data, [0.05, 0.1, 0.5])
        ruin_flat_data.sort()
        es5_idx = np.where(ruin_flat_data <= p5)
        es10_idx = np.where(ruin_flat_data <= p10)
        es5 = ruin_flat_data[es5_idx].mean()
        es10 = ruin_flat_data[es10_idx].mean()
        ruin = len(ruin_flat_data)
        min_ruin_age = ruin_flat_data[0]
        
        entry = {}
        entry["spending"] = yearly_spending
        entry["allocation"] = allocation_str
        
        entry["ruin_path_count"] = ruin
        entry["ruin_month_min"] = int(min_ruin_age)
        entry["ruin_month_median"] = float(p50)
        entry["ruin_month_es5"] = float(es5)
        entry["ruin_month_es10"] = float(es10)
        
        entry["p5_return"] = df_data["Returns"].quantile(0.05)
        entry["p10_return"] = df_data["Returns"].quantile(0.1)
        
        entry["p25_return"] = df_data["Returns"].quantile(0.25)
        entry["p50_return"] = df_data["Returns"].quantile(0.5)
        run_stats["results"][m].append(entry)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(run_stats, f, indent=4)

In [ ]:
display(Markdown("# Analysis Results"))
display(Markdown("## Simulation Parameters"))
display(Markdown(f"""| Initial NAV | Years to Simulate | Total Paths |
| :--- | :--- | :--- |
| ${initial_nav:,.2f} | {years} | {paths} |"""))

display(Markdown("## Results per model"))

for m in run_stats["results"]:
    data = run_stats["results"][m]
    df_data = pd.DataFrame(data)
    display(Markdown(f"## {m}"))
    df_data = df_data.sort_values(by=['spending', 'p10_return'], ascending=[True, False])
    df_data["ruin_rate"] = df_data["ruin_path_count"] * 100.0 / paths
    df_data["min_ruin_age"] = df_data["ruin_month_min"] / 12.0 + retirement_age
    df_data["ruin_age_p50"] = df_data["ruin_month_median"] / 12.0 + retirement_age
    df_data["ruin_age_es5"] = df_data["ruin_month_es5"] / 12.0 + retirement_age
    df_data["ruin_age_es10"] = df_data["ruin_month_es10"] / 12.0 + retirement_age
    df_data["p5_return"] *= 100.0
    df_data["p10_return"] *= 100.0
    df_data = df_data.drop(columns=[
        "ruin_path_count", "ruin_month_min", "ruin_month_median",
        "ruin_month_es5", "ruin_month_es10"])
    df_data = df_data.style.format_index(lambda x: str(x).replace("_", " ").title(), axis=1)
    df_data = df_data.format({
        "spending": "${:,.0f}",
        "ruin_rate": "{:.1f}%",
        "min_ruin_age": "{:.1f}y",
        "ruin_age_p50": "{:.1f}y",
        "ruin_age_es5": "{:.1f}y",
        "ruin_age_es10": "{:.1f}y",
        "p5_return": "{:.2f}%",
        "p10_return": "{:.2f}%",
        "p25_return": "{:.2f}x",
        "p50_return": "{:.2f}x"})
    display(df_data)